# Backtracking and Search Spaces

```{contents}
:local:
:depth: 2
```

Backtracking is depth-first search through decisions. An invariant describes the partial choice; a base case records a complete result; undo restores shared state before a sibling branch. Pruning must exclude only impossible branches. Returning a witness allows the caller to check the decision.

## Choose, Explore, Undo, and Snapshot

```{index} Choose, Explore, Undo, and Snapshot
```

At position i, chosen holds exactly the included positions from 0..i−1. Include i, recurse to i+1, undo, then exclude i. At i=n, snapshot chosen. Positions distinguish equal-valued business records; choosing position zero twice is not allowed.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var subsets=ChoicePlanning.Subsets(3);
Console.WriteLine(string.Join(" | ",subsets.Select(s=>"["+string.Join(",",s)+"]")));
Console.WriteLine($"count={subsets.Length}, empty-input results={ChoicePlanning.Subsets(0).Length}");
subsets[0][0]=99;
Console.WriteLine("independent next snapshot="+string.Join(",",subsets[1]));
public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


The include/exclude branches are disjoint and exhaustive, and each step reduces unprocessed positions, proving termination and completeness. Copying at a leaf prevents later undo from erasing saved results. There are 2^n leaves and 2^(n+1)−1 recursive calls. Total copied position entries are n·2^(n−1) for n≥1: enumeration takes Θ(n2^n) time and retained output space, even though the active stack and chosen list use only O(n). The teaching API limits n to 16 to bound retained output. Empty input has one subset, the empty subset.

## One-Use Subset Sum Needs a Two-Part State

```{index} One-Use Subset Sum Needs a Two-Part State
```

For signed invoice adjustments, ask whether some distinct positions sum to target. State (i,r) means: using only positions i..n−1, can we make remaining r? Include gives (i+1,r−value[i]); exclude gives (i+1,r). A zero remainder accepts the current subset, including the empty subset for target zero.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={8,-3,4};
var result=SubsetPlanning.FindSubset(values,5);
Console.WriteLine($"indices=[{string.Join(",",result.Indices!)}], values=[{string.Join(",",result.Indices!.Select(i=>values[i]))}]");
Console.WriteLine($"sum={result.Indices!.Sum(i=>(long)values[i])}, calls={result.Calls}");
Console.WriteLine($"empty target witness length={SubsetPlanning.FindSubset(Array.Empty<int>(),0).Indices!.Length}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


Each call advances i, so no position is reused and recursion terminates. The shared chosen list contains selected earlier positions; remaining equals target minus their sum. The implementation snapshots a successful witness and restores the list even on early success or a search-budget exception. A null witness means proven infeasibility; an empty witness is a feasible zero target. Use long sums and remaining amounts so two int values do not overflow.

## Prove Bounds Before Pruning

```{index} Prove Bounds Before Pruning
```

The rule remaining<0 is safe for nonnegative values because future additions cannot decrease the chosen sum. It fails for signed values: 8 followed by −3 makes five after temporarily overshooting. For a signed suffix define L(i) as the sum of its negative values and U(i) as the sum of its positive values. Every suffix subset sum lies in [L(i),U(i)]. Prune only if r lies outside those bounds. Being inside the interval does not guarantee feasibility.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={2,4,6,9};
var raw=SubsetPlanning.FindSubset(values,8,false,false);
var pruned=SubsetPlanning.FindSubset(values,8,true,false);
Console.WriteLine($"raw calls={raw.Calls}; pruned calls={pruned.Calls}, prunes={pruned.Prunes}");
var gap=SubsetPlanning.FindSubset(new[]{4,6},5);
Console.WriteLine($"inside bounds but feasible={gap.Indices is not null}");
var signed=SubsetPlanning.FindSubset(new[]{8,-3},5);
Console.WriteLine($"signed witness={string.Join(",",signed.Indices!)}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


Compute suffix bounds once in O(n) time/storage. The code checks success before suffix exhaustion, preserves both branches when necessary, and accepts zeros. The initial total bound check also runs in the unpruned comparison to avoid subtracting from an extreme impossible long target. Branch order affects measured calls when the search stops at its first witness; pruning has no universal speedup guarantee.

## Memoize Failed States Without Losing Choices

```{index} Memoize Failed States Without Losing Choices
```

Two different prefixes can leave the same (i,r). Their future feasibility is identical because the remaining positions and target are identical. Cache a state only after both branches fail. Do not cache only r: the same remaining amount with different inventories is a different question. Keeping only failed states avoids needing to store a prefix-dependent witness.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values=Enumerable.Repeat(2,12).ToArray();
var raw=SubsetPlanning.FindSubset(values,11,false,false);
var pruned=SubsetPlanning.FindSubset(values,11,true,false);
var memo=SubsetPlanning.FindSubset(values,11,true,true);
Console.WriteLine($"raw={raw.Calls}, bounds={pruned.Calls}, memo={memo.Calls}, hits={memo.Hits}, failed states={memo.FailedStates}");
Console.WriteLine($"feasible={memo.Indices is not null}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


For nonnegative values and target T, reachable non-pruned remainder states lie in 0..T, so there are O(n(T+1)) possible states. Signed inputs instead depend on the attainable sum range, which can be large. Expected constant-time hash lookup gives work proportional to visited states/transitions plus pruning and terminal calls. Without effective caching or pruning, worst-case branching is O(2^n), with O(n) active stack/path space. This teaching implementation accepts up to 128 positions and a default million-call budget. Exhausting it throws with feasibility unknown; it must never return false as though the instance were solved.

## Permutations Use Every Position Exactly Once

```{index} Permutations Use Every Position Exactly Once
```

At each depth choose an unused position. Maintain used[i] exactly when i appears in order, and undo both structures. Labels can repeat while positions remain distinct. This is useful when two jobs share a display name but have different IDs.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] labels={"Audit","Audit","Review"};
var orders=ChoicePlanning.Permutations(labels.Length);
foreach(var order in orders)Console.WriteLine($"IDs={string.Join(",",order)}; labels={string.Join("/",order.Select(i=>labels[i]))}");
Console.WriteLine($"position orders={orders.Length}, distinct label strings={orders.Select(o=>string.Join("/",o.Select(i=>labels[i]))).Distinct().Count()}");
Console.WriteLine($"empty-input orders={ChoicePlanning.Permutations(0).Length}");
public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


There are n! position permutations; copying each length-n order requires Θ(n·n!) time and retained output space. Active state is O(n). The API caps n at eight for materialized results. Generating unique value permutations is a different contract and would need deliberate duplicate suppression. A streamed generator can reduce retained storage, but cannot eliminate the cost of producing all requested outputs.

## Exercise

```{index} Exercise
```

Find a signed, wide-sum witness using each position once, reject an impossible target, and distinguish a budget limit from infeasibility.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={int.MaxValue,int.MaxValue,-1};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] values={int.MaxValue,int.MaxValue,-1};
var found=SubsetPlanning.FindSubset(values,2L*int.MaxValue-1);
Console.WriteLine($"sum={found.Indices!.Sum(i=>(long)values[i])}, IDs={string.Join(",",found.Indices!)}");
Console.WriteLine($"impossible={SubsetPlanning.FindSubset(values,long.MinValue).Indices is null}");
try{SubsetPlanning.FindSubset(new[]{2,2,2},3,false,false,1);}catch(InvalidOperationException){Console.WriteLine("budget exhausted: feasibility unknown");}
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


sum=4294967293, IDs=0,1,2
impossible=True
budget exhausted: feasibility unknown


```{rubric} Footnotes
```
[^1]: A bound is necessary for feasibility, not sufficient. Returning true merely because a target is between bounds is incorrect.
[^2]: Complexity includes output copying. Retaining every subset or permutation can dominate the active recursion space.